# Модель Daisyworld - Комплексное параметрическое исследование

**Цель работы:** Исследовать, как различные параметры модели влияют на
динамику численности маргариток, температуру планеты и солнечную активность.

## 1. Исследуемые параметры

В данном комплексном исследовании мы варьируем те же параметры:

| Параметр | Описание | Значения |
|----------|----------|----------|
| `max_age` | Максимальный возраст маргариток | 25, 40 шагов |
| `init_white` | Начальная доля белых маргариток | 0.2 (20%), 0.8 (80%) |

## 2. Особенности данного исследования

В отличие от предыдущего скрипта, здесь мы:
1. Используем сценарий `:ramp` — солнечная активность меняется со временем
2. Собираем данные не только о численности, но и о температуре
3. Строим комплексные графики с тремя подграфиками

## 3. Инициализация проекта и загрузка пакетов

In [1]:
using DrWatson
@quickactivate "project"

using Agents
using CairoMakie
using DataFrames
using Statistics
using JLD2
using DrWatson: dict_list, savename

include(srcdir("daisyworld.jl"))

daisyworld (generic function with 1 method)

## 4. Определение функций для сбора данных

Функции-предикаты для фильтрации агентов по виду

In [2]:
black(a) = a.breed == :black
white(a) = a.breed == :white

white (generic function with 1 method)

Данные об агентах: количество черных и белых

In [3]:
adata = [(black, count), (white, count)]

2-element Vector{Tuple{Function, typeof(count)}}:
 (Main.black, count)
 (Main.white, count)

Функция для вычисления средней температуры по всей планете

In [4]:
temperature(model) = Statistics.mean(model.temperature)

temperature (generic function with 1 method)

Данные о модели: температура и солнечная активность

In [5]:
mdata = [temperature, :solar_luminosity]

2-element Vector{Any}:
 temperature (generic function with 1 method)
 :solar_luminosity

## 5. Словарь параметров для исследования

In [6]:
param_dict = Dict(
    :griddim => (30, 30),
    :max_age => [25, 40],           # два варианта максимального возраста
    :init_white => [0.2, 0.8],       # два варианта начальной доли белых
    :init_black => 0.2,
    :albedo_white => 0.75,
    :albedo_black => 0.25,
    :surface_albedo => 0.4,
    :solar_change => 0.005,
    :solar_luminosity => 1.0,
    :scenario => :ramp,               # важно: используем сценарий с изменением активности
    :seed => 165,
)

Dict{Symbol, Any} with 11 entries:
  :solar_luminosity => 1.0
  :griddim          => (30, 30)
  :scenario         => :ramp
  :max_age          => [25, 40]
  :surface_albedo   => 0.4
  :init_white       => [0.2, 0.8]
  :albedo_white     => 0.75
  :init_black       => 0.2
  :solar_change     => 0.005
  :albedo_black     => 0.25
  :seed             => 165

Генерируем все комбинации параметров

In [7]:
params_list = dict_list(param_dict)

println("="^60)
println("МОДЕЛЬ DAISYWORLD - КОМПЛЕКСНОЕ ПАРАМЕТРИЧЕСКОЕ ИССЛЕДОВАНИЕ")
println("="^60)
println("\n📊 Всего комбинаций параметров: ", length(params_list))
println("\nИсследуемые комбинации:")
for (i, params) in enumerate(params_list)
    println("   $i. max_age = $(params[:max_age]), init_white = $(params[:init_white])")
end
println("\n🔄 Сценарий: :ramp (солнечная активность меняется со временем)")

МОДЕЛЬ DAISYWORLD - КОМПЛЕКСНОЕ ПАРАМЕТРИЧЕСКОЕ ИССЛЕДОВАНИЕ

📊 Всего комбинаций параметров: 4

Исследуемые комбинации:
   1. max_age = 25, init_white = 0.2
   2. max_age = 25, init_white = 0.8
   3. max_age = 40, init_white = 0.2
   4. max_age = 40, init_white = 0.8

🔄 Сценарий: :ramp (солнечная активность меняется со временем)


## 6. Запуск для всех комбинаций параметров

Создаем директорию для сохранения результатов

In [8]:
param_plots_dir = plotsdir("daisyworld-luminosity_param")
mkpath(param_plots_dir)

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab03/project/plots/daisyworld-luminosity_param"

Таблица для сводных результатов

In [26]:
# -------------------------------------------------------------------
# 4. ЗАПУСК ДЛЯ ВСЕХ КОМБИНАЦИЙ ПАРАМЕТРОВ
# -------------------------------------------------------------------

# Создаем директорию для сохранения графиков
param_plots_dir = plotsdir("daisyworld-luminosity_param")
mkpath(param_plots_dir)

# Массив для хранения результатов ВСЕХ запусков
results_summary = []

# ЦИКЛ ПО ВСЕМ КОМБИНАЦИЯМ ПАРАМЕТРОВ
for (idx, params) in enumerate(params_list)
    println("\n" * "-"^60)
    println("🔄 Запуск $idx/$(length(params_list))")
    println("   Параметры: max_age = $(params[:max_age]), init_white = $(params[:init_white])")
    
    # ---------------------------------------------------------------
    # 4.1 Создание модели с текущими параметрами
    # ---------------------------------------------------------------
    model = daisyworld(; params...)
    
    # ---------------------------------------------------------------
    # 4.2 Запуск симуляции и сбор данных
    # ---------------------------------------------------------------
    temperature(model) = Statistics.mean(model.temperature)
    mdata = [temperature, :solar_luminosity]
    
    agent_df, model_df = run!(model, 1000; adata = adata, mdata = mdata)
    
    # ---------------------------------------------------------------
    # 4.3 Сохранение сводных результатов
    # ---------------------------------------------------------------
    push!(results_summary, Dict(
        :max_age => params[:max_age],
        :init_white => params[:init_white],
        :mean_black => mean(agent_df[!, :count_black]),
        :mean_white => mean(agent_df[!, :count_white]),
        :mean_temperature => mean(model_df[!, :temperature]),
        :final_black => agent_df[!, :count_black][end],
        :final_white => agent_df[!, :count_white][end]
    ))
    
    # ---------------------------------------------------------------
    # 4.4 Создание комплексного графика
    # ---------------------------------------------------------------
    fig = Figure(size = (600, 600))
    
    # График 1: Численность маргариток
    ax1 = Axis(fig[1, 1],
        ylabel = "Количество маргариток",
        title = "Динамика численности (max_age=$(params[:max_age]), init_white=$(params[:init_white]))"
    )
    
    black_line = lines!(ax1, agent_df[!, :time], agent_df[!, :count_black],
        color = :red, linewidth = 2, label = "Черные")
    white_line = lines!(ax1, agent_df[!, :time], agent_df[!, :count_white],
        color = :blue, linewidth = 2, label = "Белые")
    
    axislegend(ax1, [black_line, white_line], ["Черные", "Белые"], position = :rt)
    
    # График 2: Температура
    ax2 = Axis(fig[2, 1],
        ylabel = "Средняя температура, °C"
    )
    lines!(ax2, model_df[!, :time], model_df[!, :temperature],
        color = :red, linewidth = 2)
    
    # График 3: Солнечная активность
    ax3 = Axis(fig[3, 1],
        xlabel = "Время, шаги",
        ylabel = "Солнечная активность"
    )
    lines!(ax3, model_df[!, :time], model_df[!, :solar_luminosity],
        color = :red, linewidth = 2)
    
    # Скрываем подписи на верхних графиках
    for ax in (ax1, ax2)
        ax.xticklabelsvisible = false
    end
    
    # ---------------------------------------------------------------
    # 4.5 Сохранение графика
    # ---------------------------------------------------------------
    plt_name = savename("daisy-luminosity", params, "png", digits=5)
    save(joinpath(param_plots_dir, plt_name), fig)
    println("   ✅ График сохранён: ", joinpath(param_plots_dir, plt_name))
    
end  # ← КОНЕЦ ЦИКЛА for - ВАЖНО!

# -------------------------------------------------------------------
# 5. АНАЛИЗ РЕЗУЛЬТАТОВ (ЭТОТ КОД ПОСЛЕ ЦИКЛА)
# -------------------------------------------------------------------
println("\n" * "="^60)
println("📊 СВОДНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ")
println("="^60)

# Создаем сводную таблицу из сохраненных результатов
results_df = DataFrame(results_summary)
println("\nСводная таблица результатов:")
println(results_df)

# Сохраняем результаты
@save datadir("daisyworld-luminosity_param_results.jld2") results_df

# Анализируем результаты (используем results_summary, а НЕ params!)
println("\n📈 Анализ влияния параметров:")
for row in results_summary
    println("\nПараметры: max_age=$(row[:max_age]), init_white=$(row[:init_white])")
    println("   - Средняя численность черных: ", round(row[:mean_black], digits=1))
    println("   - Средняя численность белых: ", round(row[:mean_white], digits=1))
    println("   - Средняя температура: ", round(row[:mean_temperature], digits=1))
end


------------------------------------------------------------
🔄 Запуск 1/4
   Параметры: max_age = 25, init_white = 0.2
   ✅ График сохранён: /afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab03/project/plots/daisyworld-luminosity_param/daisy-luminosity_albedo_black=0.25_albedo_white=0.75_init_black=0.2_init_white=0.2_max_age=25_scenario=ramp_seed=165_solar_change=0.005_solar_luminosity=1.0_surface_albedo=0.4.png

------------------------------------------------------------
🔄 Запуск 2/4
   Параметры: max_age = 25, init_white = 0.8
   ✅ График сохранён: /afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab03/project/plots/daisyworld-luminosity_param/daisy-luminosity_albedo_black=0.25_albedo_white=0.75_init_black=0.2_init_white=0.8_max_age=25_scenario=ramp_seed=165_solar_change=0.005_solar_luminosity=1.0_surface_albedo=0.4.png

-----------------------------------------------------

## 7. Сводный анализ результатов

In [19]:
println("\n" * "="^60)
println("📊 СВОДНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ")
println("="^60)


📊 СВОДНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ


Создаем сводную таблицу

In [20]:
results_df = DataFrame(results_summary)
println("\nСводная таблица результатов:")
println(results_df)


Сводная таблица результатов:
0×0 DataFrame


Сохраняем результаты в JLD2

In [21]:
@save datadir("daisyworld-luminosity_param_results.jld2") results_df

println("\n📈 Анализ влияния параметров на комплексную динамику:")
println("-"^60)
for row in eachrow(results_df)
    println("\n📌 Параметры: max_age = $(row.max_age), init_white = $(row.init_white)")
    println("   • Средняя численность черных: ", round(row.mean_black, digits=1))
    println("   • Средняя численность белых: ", round(row.mean_white, digits=1))
    println("   • Средняя температура: ", round(row.mean_temperature, digits=1), "°C")
    println("   • Диапазон солнечной активности: ",
        round(row.min_luminosity, digits=2), " - ", round(row.max_luminosity, digits=2))
end

println("\n" * "="^60)
println("📌 ВЫВОДЫ ПО КОМПЛЕКСНОМУ ИССЛЕДОВАНИЮ:")
println("   • При сценарии :ramp система успевает адаптироваться к изменениям")
println("   • Более высокая продолжительность жизни (max_age=40) увеличивает стабильность")
println("   • Начальное соотношение видов влияет на амплитуду колебаний температуры")
println("   • Система демонстрирует саморегуляцию во всех исследованных случаях")
println("="^60)
println("\n✅ Комплексное параметрическое исследование завершено!")


📈 Анализ влияния параметров на комплексную динамику:
------------------------------------------------------------

📌 ВЫВОДЫ ПО КОМПЛЕКСНОМУ ИССЛЕДОВАНИЮ:
   • При сценарии :ramp система успевает адаптироваться к изменениям
   • Более высокая продолжительность жизни (max_age=40) увеличивает стабильность
   • Начальное соотношение видов влияет на амплитуду колебаний температуры
   • Система демонстрирует саморегуляцию во всех исследованных случаях

✅ Комплексное параметрическое исследование завершено!
